# 🧪 Fine-Tuning Eficiente de um Modelo de Linguagem com LoRA (PEFT)

Este tutorial conduz você, passo a passo, pelo processo de **ajuste fino (fine-tuning)** de um modelo de linguagem causal (ex.: `distilgpt2`) utilizando **LoRA** (*Low-Rank Adaptation*), uma das técnicas de **PEFT** (*Parameter-Efficient Fine-Tuning*).  
Ao final, você compreenderá os fundamentos matemáticos por trás do método e saberá como adaptar um modelo grande com muito menos recursos computacionais.

**O que você vai aprender:**
- A diferença entre *full fine-tuning* e *PEFT*.
- A ideia central do LoRA (matrizes de baixo posto).
- Como preparar dados, configurar, treinar e testar um modelo com LoRA.
- Comparar a qualidade das respostas antes e depois do ajuste fino.

## 📚 1. Por que Fine-Tuning Eficiente?

Modelos de linguagem modernos possuem bilhões de parâmetros. Atualizar **todos** os pesos durante o treinamento (*full fine-tuning*) exige:
- GPUs com dezenas de GB de memória.
- Armazenamento de uma cópia completa do modelo para cada tarefa.

**PEFT (Parameter-Efficient Fine-Tuning)** resolve esse problema treinando apenas um pequeno conjunto de **novos parâmetros**, mantendo o modelo base congelado.  

### 🔹 LoRA (Low-Rank Adaptation)
A hipótese do LoRA é que as atualizações dos pesos durante o fine-tuning possuem uma **estrutura de baixo posto** (*low intrinsic rank*).  
Assim, em vez de aprender a matriz completa de atualização $\Delta W \in \mathbb{R}^{d \times k}$, aprendemos duas matrizes menores:

$$\Delta W = B \cdot A$$

onde:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (o **rank** da adaptação)

O número de parâmetros treináveis cai de $d \times k$ para $r \times (d + k)$, uma redução drástica quando $r$ é pequeno.

### 🔹 Como isso é usado na prática?
Durante o treinamento, a saída de uma camada linear original $h = W x$ é modificada para:

$$h = W x + \Delta W x = W x + B A x$$

A matriz $A$ é inicializada com uma distribuição gaussiana e $B$ com zeros, de forma que no início $\Delta W = 0$.  
Um fator de escala $\alpha$ controla a intensidade da adaptação; frequentemente a atualização é escalada por $\frac{\alpha}{r}$:

$$h = W x + \frac{\alpha}{r} B A x$$

Após o treinamento, podemos **fundir** (*merge*) os pesos adaptados ao modelo original: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, eliminando qualquer custo extra na inferência.

Neste notebook, usaremos a biblioteca `peft` (Hugging Face) para aplicar LoRA ao `distilgpt2`.

## 📦 2. Requisitos

Execute o comando abaixo para instalar as dependências necessárias (descomente a linha caso ainda não estejam instaladas):

In [6]:
#!pip install transformers datasets peft accelerate torch

Importe os módulos que serão utilizados ao longo do processo:

In [7]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

## 🔢 3. Carregar e Preparar o Dataset

Utilizaremos um arquivo `dataset.jsonl` onde cada linha contém uma instrução (`instruction`) e a saída desejada (`output`).  
Vamos converter cada exemplo em uma única string no formato:
```
Instruction: <instrução>
Output: <saída>
```
e depois dividir o conjunto em treino (80%) e validação (20%).

In [8]:
def convert_to_hf_format(example):
    """Combina instrução e saída em um único texto."""
    return {
        "text": f"Instruction: {example['instruction']}\nOutput: {example['output']}"
    }

# Carrega o arquivo JSON Lines
dataset = load_dataset('json', data_files='dataset.jsonl')
# Aplica a conversão
dataset = dataset.map(convert_to_hf_format)
# Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2
    })
})


## 🤖 4. Carregar o Modelo Pré-Treinado e o Tokenizador

Vamos carregar o `distilgpt2` – uma versão menor e mais rápida do GPT-2, ideal para experimentação.  
Como o tokenizador original não define um `pad_token`, usaremos o `eos_token` no lugar.

In [9]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 6916.49it/s]


Modelo carregado: distilgpt2


## 🧪 5. Inferência ANTES do Fine-Tuning

Antes de qualquer treinamento, vamos ver como o modelo base responde a uma pergunta que está no nosso dataset.  
Isso servirá como **linha de base** para compararmos com o modelo ajustado.

In [10]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # ativa amostragem para usar temperatura
        temperature=0.7
    )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a parte após "Output:"
    resposta = full_output.split("Output:")[-1].strip()
    return resposta

# Exemplo de instrução (deve existir no dataset.jsonl)
test_instruction = "How do I activate cruise control?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta base: 


> **Observação:** O modelo base provavelmente gerará um texto genérico ou sem relação direta com a instrução, pois ainda não foi adaptado ao nosso domínio.

## ✂️ 6. Tokenização do Dataset

Transformamos os textos em sequências de tokens que o modelo pode processar.  
Usaremos `padding="max_length"` e `truncation=True` para garantir que todas as amostras tenham o mesmo comprimento (128 tokens).

In [24]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 2
    })
})


## 🔧 7. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa técnicas como *gradient checkpointing* e ajusta a arquitetura para treinamento eficiente.  
É essencial quando se utiliza quantização (QLoRA), mas também é recomendada mesmo sem quantização para melhor gerenciamento de memória.

In [12]:
# A partir daqui, usaremos uma cópia do modelo base para o fine-tuning
model = base_model  # poderia também ser uma nova instância
model = prepare_model_for_kbit_training(model)

## 🧩 8. Configurar e Injetar LoRA

Agora definimos a configuração do LoRA:
- **r**: posto das matrizes de adaptação (quanto maior, mais capacidade, porém mais parâmetros).
- **lora_alpha**: fator de escala $\alpha$ (a atualização será multiplicada por $\alpha / r$).
- **target_modules**: os módulos do transformer onde aplicaremos LoRA. No GPT-2, `c_attn` e `c_proj` são as projeções de atenção.
- **lora_dropout**: dropout aplicado às matrizes LoRA para regularização.
- **bias**: "none" significa que não treinamos os vieses.
- **task_type**: como é um modelo de linguagem causal, usamos `CAUSAL_LM`.

Em seguida, criamos o modelo PEFT com `get_peft_model`, que insere os adaptadores LoRA e **congela** o resto do modelo.

In [13]:
lora_config = LoraConfig(
    r=16,                    # rank da decomposição
    lora_alpha=32,           # escala alpha
    target_modules=["c_attn", "c_proj"],  # camadas alvo
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False     # False = modo treinamento
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/home/aluno/TestAPI/.venv-1/lib/python3.13/site-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 811,008 || all params: 82,723,584 || trainable%: 0.9804


✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 🧱 9. Data Collator para Modelagem Causal

O `DataCollatorForLanguageModeling` prepara os lotes para o treinamento de linguagem causal (sem *masked language modeling*).  
Ele automaticamente desloca os rótulos para que a tarefa seja prever o próximo token.

In [14]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## ⚙️ 10. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.  
Como nosso dataset é pequeno, usaremos 100 épocas e uma taxa de aprendizado relativamente alta (`1e-3`).  
O `eval_steps` controla a frequência da avaliação no conjunto de validação.

In [15]:
training_args = TrainingArguments(
    output_dir="./results",          # diretório de saída
    eval_strategy="steps",
    eval_steps=100,
    learning_rate=1e-3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=100,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",               # desabilita logging para wandb/mlflow
)

## 🏋️ 11. Inicializar o Trainer

O `Trainer` do Hugging Face orquestra todo o ciclo de treinamento, avaliação e salvamento.

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## 🚀 12. Treinar o Modelo

Iniciamos o treinamento. Acompanhe a perda (*loss*) nos logs – ela deve diminuir ao longo das épocas.

In [17]:
trainer.train()

/home/aluno/TestAPI/.venv-1/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,0.268742,4.686073
200,0.101252,4.959034


/home/aluno/TestAPI/.venv-1/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=200, training_loss=0.7066423678398133, metrics={'train_runtime': 244.0199, 'train_samples_per_second': 3.278, 'train_steps_per_second': 0.82, 'total_flos': 26627958374400.0, 'train_loss': 0.7066423678398133, 'epoch': 100.0})

## 💾 13. Salvar o Modelo Ajustado e o Tokenizador

Ao final do treinamento, salvamos os pesos LoRA (apenas os adaptadores) e o tokenizador.

In [21]:
model.save_pretrained("lora_finetuned_model")
tokenizer.save_pretrained("distilgpt2_tokenizer")

('distilgpt2_tokenizer/tokenizer_config.json',
 'distilgpt2_tokenizer/tokenizer.json')

## 💻 14. Inferência APÓS o Fine-Tuning

Agora carregamos o modelo ajustado e comparamos sua resposta com a versão base, usando **exatamente a mesma instrução**.

In [19]:
# Carrega o modelo fine-tunado (apenas os adaptadores LoRA)
finetuned_model = AutoModelForCausalLM.from_pretrained("lora_finetuned_model")
finetuned_tokenizer = AutoTokenizer.from_pretrained("distilgpt2_tokenizer")

# Garante que o token de padding esteja definido
if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

Loading weights: 100%|██████████| 36/36 [00:00<00:00, 25544.74it/s]


In [22]:
print("=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response(finetuned_model, finetuned_tokenizer, test_instruction)}")

=== DEPOIS DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta ajustada: To activate the cabin air filter on a 2020 Chevy Silverado:
1. Turn ignition to ON (engine off)
2. Press the 'Options' button on the steering wheel
3. Scroll to 'Oil Life' display
4.


## 📊 15. Comparação e Conclusão

- **Antes do fine-tuning:** o modelo base não conhecia nosso domínio; sua resposta era genérica ou incoerente.
- **Depois do fine-tuning:** com apenas uma fração dos parâmetros treinados (via LoRA), o modelo aprendeu a estrutura desejada e gera respostas alinhadas com os exemplos fornecidos.

Esse é o poder do **PEFT**: adaptar grandes modelos de forma rápida, barata e com resultados surpreendentes.

### 📌 Resumo dos conceitos-chave

| Conceito | Descrição |
|----------|-----------|
| **Full fine-tuning** | Atualiza todos os pesos do modelo. |
| **PEFT** | Atualiza apenas um pequeno número de parâmetros novos. |
| **LoRA** | Decompõe a atualização $\Delta W$ em $B A$, com $r \ll \min(d,k)$. |
| **r** | Posto da decomposição – controla a capacidade da adaptação. |
| **$\alpha$** | Fator de escala que ajusta a intensidade da adaptação. |
| **Target modules** | Camadas onde os adaptadores LoRA são inseridos. |

Agora você pode experimentar com outros valores de `r`, `lora_alpha`, ou até mesmo outros modelos!